# # M07 - Supervised fine-tuning a small language model
#
# **Hardware:** CPU (about 30 seconds of training) · **Time:** 3-4 hours ·
# **Prerequisite:** M06
#
# Pre-training teaches a model to continue text. It does not teach the model to
# answer a question in a particular format. Supervised fine-tuning (SFT) does
# that with pairs of `(prompt, answer)`, a chat template, and one crucial detail:
# the **prompt tokens are masked out of the loss**.
#
# This notebook trains a base model on the Northwind FAQ corpus, fine-tunes it on
# 120 instruction pairs, and measures the change on 16 held-out questions. Every
# number is produced here; nothing is quoted from a paper.
#
# **You will produce:** a base checkpoint, a tuned checkpoint, a metric table,
# and generation samples from both models.

# ## 1. Setup and the dataset
#
# The FAQ data is synthetic (CC0-1.0), generated from a fixed seed. It has eight
# topics and three splits: 120 training pairs, 16 validation, 16 test. Splits are
# disjoint by **question template**, not by topic - every topic appears in all
# three splits. That distinction matters: a high test score here means the model
# handles rephrased questions about known topics, not that it generalises to a
# new subject area.

In [1]:
from pathlib import Path
import copy
import sys
import time

import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "projects").is_dir())
sys.path.insert(0, str(ROOT / "projects" / "llm-fine-tuning"))

import llm_core as lc

lc.ensure_datasets(quiet=True)
train_pairs = lc.load_faq("train")
validation_pairs = lc.load_faq("validation")
test_pairs = lc.load_faq("test")
print("train/validation/test:", len(train_pairs), len(validation_pairs), len(test_pairs))
print("topics:", sorted({row["topic"] for row in train_pairs}))

train_questions = {row["question"] for row in train_pairs}
test_questions = {row["question"] for row in test_pairs}
print("question overlap between train and test:", len(train_questions & test_questions))
assert len(train_questions & test_questions) == 0

for row in test_pairs[:3]:
    print(f"\n[{row['topic']}] {row['question']}\n  -> {row['answer']}")

train/validation/test: 120 16 16
topics: ['accessories', 'orders', 'payment', 'repairs', 'returns', 'shipping', 'sizing', 'warranty']
question overlap between train and test: 0

[returns] What happens if the cycling computer does not fit?
  -> Unused items can be returned within 30 days with the receipt. Refunds are issued to the original payment method within 5 to 7 business days.

[orders] My u-lock never arrived; what should I do?
  -> Orders can be changed or cancelled before they ship. Order status is shown in the account page, and pickup orders are held for 7 days.

[repairs] What does a standard service include for a u-lock?
  -> Repair appointments are booked online or at the service desk. A standard tune-up costs 60 USD, and most repairs are ready in 2 to 3 days.


# ## 2. Build the base model
#
# The base model is a character-level transformer trained on the FAQ text with no
# template at all: just questions and answers, newline separated. It learns the
# vocabulary and style of the domain. It has never seen the strings `User:` or
# `Assistant:`, so it cannot follow a chat instruction yet.
#
# Tokenization is shared between pre-training and fine-tuning. Changing the
# tokenizer between stages would invalidate the embedding table, which is another
# reason the tokenizer is part of the model artifact.

In [2]:
faq_text = "\n".join(f"{row['question']}\n{row['answer']}" for row in train_pairs)
tokenizer = lc.CharTokenizer.from_text(
    lc.load_corpus() + "\n" + faq_text + "\nUser: Assistant:\n"
)
print("vocab size:", len(tokenizer))

base = lc.build_model(len(tokenizer), block_size=160, n_layer=2, n_head=4, n_embd=96, dropout=0.1)
print("parameters:", base.parameter_count())

started = time.perf_counter()
base_history = lc.train_lm(
    base,
    tokenizer.encode(faq_text),
    steps=400,
    batch_size=8,
    block_size=160,
    lr=3e-3,
    seed=1,
    warmup=20,
    log_every=100,
)
print(base_history.summary(), "wall", round(time.perf_counter() - started, 1), "s")
print("base perplexity on its own text:", round(lc.perplexity(base_history.losses[-1]), 2))
assert base_history.losses[-1] < base_history.losses[0]

vocab size: 56
parameters: 244608


{'steps': 400, 'first_loss': 4.0392, 'last_loss': 1.2014, 'seconds': 16.75, 'tokens': 512000, 'tokens_per_second': 30568, 'evals': []} wall 17.7 s
base perplexity on its own text: 3.32


# ## 3. The chat template
#
# A chat template is a contract between training and inference. The model must
# see exactly the same layout in both. Ours is deliberately minimal:
#
# ```text
# User: <question>
# Assistant: <answer>
# ```
#
# At inference we stop after `Assistant:` and let the model continue. If the
# template differed by even a space between training and serving, the model's
# learned behaviour would not transfer.

In [3]:
example = train_pairs[0]
print(lc.format_prompt(example["question"]))
print(lc.format_example(example["question"], example["answer"]))
print("\nprompt preview:", repr(lc.format_prompt("Is installation included?")))

User: Is installation included with a helmet?
Assistant:
User: Is installation included with a helmet?
Assistant: Accessories are compatible when the product page lists your bike model. Helmets, lights, and locks can be returned within 30 days if unused.


prompt preview: 'User: Is installation included?\nAssistant:'


# ## 4. Prompt masking, token by token
#
# Here is the detail most tutorials skip. We tokenise prompt and answer together,
# shift the labels by one so position `t` predicts token `t+1`, and set every
# prompt label to `-100`. PyTorch's cross-entropy ignores `-100`.
#
# Without the shift, the model is trained to copy the token it is currently
# reading: the loss collapses to near zero and generation degenerates, because at
# inference the next token does not exist yet. The cell below checks the shift
# explicitly.

In [4]:
sft_examples = lc.build_examples(train_pairs, tokenizer, block_size=160)
test_examples = lc.build_examples(test_pairs, tokenizer, block_size=160)
first = sft_examples[0]

print("input length:", len(first.input_ids), "answer tokens:", first.answer_tokens)
print("prompt share of tokens (masked):", lc.prompt_fraction(sft_examples))
print("\nposition | input char | label char")
for index in range(0, min(len(first.input_ids), 24)):
    label = first.labels[index]
    print(f"{index:7d} | {tokenizer.decode([first.input_ids[index]])!r:10} | "
          f"{'IGNORED' if label == -100 else repr(tokenizer.decode([label]))}")

for example in sft_examples[:5]:
    assert all(label == -100 for label in example.labels[: example.prompt_tokens - 1])
    for index, label in enumerate(example.labels[:-1]):
        if label != -100:
            assert label == example.input_ids[index + 1]
print("\nmasking and shift verified")

input length: 159 answer tokens: 104
prompt share of tokens (masked): 0.378

position | input char | label char
      0 | 'U'        | IGNORED
      1 | 's'        | IGNORED
      2 | 'e'        | IGNORED
      3 | 'r'        | IGNORED
      4 | ':'        | IGNORED
      5 | ' '        | IGNORED
      6 | 'I'        | IGNORED
      7 | 's'        | IGNORED
      8 | ' '        | IGNORED
      9 | 'i'        | IGNORED
     10 | 'n'        | IGNORED
     11 | 's'        | IGNORED
     12 | 't'        | IGNORED
     13 | 'a'        | IGNORED
     14 | 'l'        | IGNORED
     15 | 'l'        | IGNORED
     16 | 'a'        | IGNORED
     17 | 't'        | IGNORED
     18 | 'i'        | IGNORED
     19 | 'o'        | IGNORED
     20 | 'n'        | IGNORED
     21 | ' '        | IGNORED
     22 | 'i'        | IGNORED
     23 | 'n'        | IGNORED

masking and shift verified


# ## 5. Fine-tune
#
# The SFT learning rate is lower than pre-training (5e-4 instead of 3e-3) and the
# run is shorter (120 steps). Fine-tuning is a small adjustment to an existing
# model, not a second pre-training run. A high learning rate here destroys the
# base model's language ability - the "catastrophic forgetting" measured in
# section 7.

In [5]:
tuned = copy.deepcopy(base)
started = time.perf_counter()
sft_history = lc.train_sft(
    tuned,
    sft_examples,
    steps=120,
    batch_size=8,
    lr=5e-4,
    seed=2,
    pad_id=tokenizer.pad_id,
    warmup=10,
    log_every=40,
    eval_examples=test_examples,
    eval_every=60,
)
print(sft_history.summary(), "wall", round(time.perf_counter() - started, 1), "s")
print("logged losses:", [round(loss, 3) for loss in sft_history.losses])
assert sft_history.losses[-1] < sft_history.losses[0]

{'steps': 120, 'first_loss': 1.2177, 'last_loss': 0.6513, 'seconds': 5.15, 'tokens': 95137, 'tokens_per_second': 18463, 'evals': [(60, 0.394), (120, 0.3146)]} wall 5.2 s
logged losses: [1.218, 0.731, 0.596, 0.651]


# ## 6. Measure: base vs fine-tuned
#
# Three numbers on the 16 held-out questions:
#
# * **answer loss** - cross-entropy on answer tokens only (lower is better).
# * **answer accuracy** - share of answer tokens where the correct token ranks
#   first, given the true prefix.
# * **first-12 accuracy** - the same, restricted to the first twelve answer
#   tokens, which is where the response format is decided.
#
# Teacher-forced accuracy is the honest primary metric at this scale. Free-running
# generation compounds one wrong character into many, so exact match is near zero
# for both models and says more about decoding than about learning.

In [6]:
def measure(model, label):
    return {
        "model": label,
        "answer_loss": round(lc.estimate_sft_loss(model, test_examples, pad_id=tokenizer.pad_id), 3),
        "accuracy": round(lc.masked_accuracy(model, test_examples, pad_id=tokenizer.pad_id), 3),
        "first_12": round(lc.masked_accuracy(model, test_examples, pad_id=tokenizer.pad_id, first_k=12), 3),
    }


rows = [measure(base, "base"), measure(tuned, "SFT")]
print(lc.format_table(rows, ["model", "answer_loss", "accuracy", "first_12"]))
assert rows[1]["answer_loss"] < rows[0]["answer_loss"]

model  answer_loss  accuracy  first_12
-----  -----------  --------  --------
base   1.039        0.688     0.411   
SFT    0.315        0.905     0.818   


# ## 7. Generation, and what it still gets wrong
#
# Numbers are not the whole story. The samples below show the qualitative change:
# the base model continues the prompt as if it were plain text, while the tuned
# model emits an answer-shaped response. Both are still imperfect - a 240k
# parameter character model is not a chat assistant. Treat the samples as
# evidence of format learning, not of factual reliability.
#
# Notice also that the tuned model's free-running output drifts after the first
# sentence. That is the model equivalent of knowing the first line of a speech
# and improvising the rest.

In [7]:
def show(model, question, temperature=0.0):
    answer = lc.generate_answer(model, tokenizer, question, max_new_tokens=110, temperature=temperature)
    return answer


for row in test_pairs[:3]:
    print("\nQ:", row["question"])
    print("  base :", repr(show(base, row["question"])))
    print("  tuned:", repr(show(tuned, row["question"])))
    print("  ref  :", repr(row["answer"]))

f1_base = sum(lc.word_f1(show(base, row["question"]), row["answer"]) for row in test_pairs[:8]) / 8
f1_tuned = sum(lc.word_f1(show(tuned, row["question"]), row["answer"]) for row in test_pairs[:8]) / 8
print(f"\nword F1 on 8 questions: base {f1_base:.3f} -> tuned {f1_tuned:.3f}")
print("(bag-of-words overlap only; a weak metric, reported for transparency)")


Q: What happens if the cycling computer does not fit?


  base : 'and rs and rd rs and rd rd rd rs ard rdy rs rs ardelivery takes the Stan day tardelivery takes take 3 5 to bu'


  tuned: 'Orders can be canged celefore be be be p. Refun ar ar are are are ad ard ard livery theskesheshe A Orestacaca'
  ref  : 'Unused items can be returned within 30 days with the receipt. Refunds are issued to the original payment method within 5 to 7 business days.'

Q: My u-lock never arrived; what should I do?


  base : 're re re re reacand rd rd rd rdeardelivery takes takes 3 to wice the the 30 wif rececec.'


  tuned: 'Orders can canced belefore they ship. Order status is ishown in the acccount page, and and and lift an p p pa'
  ref  : 'Orders can be changed or cancelled before they ship. Order status is shown in the account page, and pickup orders are held for 7 days.'

Q: What does a standard service include for a u-lock?


  base : 'can can can cart cart cartift comute?'


  tuned: 'Orders can bed cangelefore whe p. s 2 o o o a pm pm a busine the sin day. Ordatus is ishown in the the accccc'
  ref  : 'Repair appointments are booked online or at the service desk. A standard tune-up costs 60 USD, and most repairs are ready in 2 to 3 days.'



word F1 on 8 questions: base 0.087 -> tuned 0.258
(bag-of-words overlap only; a weak metric, reported for transparency)


# ## 8. Forgetting
#
# Fine-tuning can damage what pre-training learned. We measure it directly: the
# loss on the original plain-text FAQ corpus, with no template, before and after
# SFT. A tuned model that has forgotten how to model plain text will score worse
# here even though it is better at answering.
#
# Some forgetting is expected and acceptable. The question is whether the model
# can still perform the capability you are shipping. If plain-text modelling
# matters, mix a fraction of pre-training data into the SFT batch - the standard
# rehearsal remedy.

In [8]:
corpus_tokens = tokenizer.encode(faq_text)
loss_before = lc.estimate_loss(base, corpus_tokens, batch_size=8, block_size=160, batches=6)
loss_after = lc.estimate_loss(tuned, corpus_tokens, batch_size=8, block_size=160, batches=6)
print(f"plain-text FAQ loss: base {loss_before:.3f} -> tuned {loss_after:.3f}")
print("change:", round(loss_after - loss_before, 3))

plain-text FAQ loss: base 0.885 -> tuned 1.072
change: 0.187


# ## 9. Save both checkpoints
#
# Saving the base as well as the tuned model is not sentimentality. A regression
# in production is diagnosed by comparing against the previous model, and an
# adapter or a rollback needs the exact base weights it was trained from.

In [9]:
output_dir = ROOT / "notebooks" / "llm-fine-tuning" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
for name, model in [("m07_base.pt", base), ("m07_sft.pt", tuned)]:
    path = output_dir / name
    torch.save(
        {"state_dict": model.state_dict(), "config": model.config.as_dict(), "vocab": tokenizer.vocab},
        path,
    )
    print("saved", name, round(path.stat().st_size / 1024, 1), "KiB")

saved m07_base.pt 966.2 KiB
saved m07_sft.pt 965.6 KiB


# ## 10. Validation and exercises
#
# **Exercises**
#
# 1. Train SFT on only 24 pairs. Report accuracy and loss on the same test set.
#    How much of the gain survives with one fifth of the data?
# 2. Remove prompt masking (set labels equal to inputs everywhere) and retrain.
#    Explain the resulting loss and the generated text.
# 3. Replace the template with `### Question` / `### Answer`. What breaks if you
#    evaluate with the old template?
# 4. Add 10% plain FAQ text to every SFT batch. Does forgetting shrink? What
#    happens to answer accuracy?
# 5. Optional GPU path: the same code runs on CUDA by moving the model and the
#    tensors; report tokens/second, not a quality change.

In [10]:
checks = {
    "splits are disjoint": len(train_questions & test_questions) == 0,
    "prompt is masked": all(
        label == -100 for label in sft_examples[0].labels[: sft_examples[0].prompt_tokens - 1]
    ),
    "labels are shifted": all(
        label == example.input_ids[index + 1]
        for example in sft_examples[:3]
        for index, label in enumerate(example.labels[:-1])
        if label != -100
    ),
    "SFT loss decreased": sft_history.losses[-1] < sft_history.losses[0],
    "held-out loss improved": rows[1]["answer_loss"] < rows[0]["answer_loss"],
    "held-out accuracy improved": rows[1]["accuracy"] > rows[0]["accuracy"],
    "checkpoints written": all((output_dir / name).exists() for name in ("m07_base.pt", "m07_sft.pt")),
}
for name, passed in checks.items():
    print(f"{'ok' if passed else 'FAIL'} - {name}")
    assert passed, name
print("\nM07 validation passed.")

ok - splits are disjoint
ok - prompt is masked
ok - labels are shifted
ok - SFT loss decreased
ok - held-out loss improved
ok - held-out accuracy improved
ok - checkpoints written

M07 validation passed.
